In [1]:
print("Notebook 2 — Nettoyage et transformation des données")

Notebook 2 — Nettoyage et transformation des données


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark =  (SparkSession.builder
            .appName("TradeCorp ETL")
            .getOrCreate()
        )


In [3]:
path_raw = '../data/raw/'

df_categories = spark.read.csv(f"{path_raw}categories.csv",header=True,inferSchema=True)
df_customers = spark.read.csv(f"{path_raw}customers.csv",header=True,inferSchema=True)
df_employees =  spark.read.csv(f"{path_raw}employees.csv",header=True,inferSchema=True)
df_order_details = spark.read.csv(f"{path_raw}order_details.csv",header=True,inferSchema=True)
df_orders = spark.read.csv(f"{path_raw}orders.csv",header=True,inferSchema=True)
df_products = spark.read.csv(f"{path_raw}products.csv",header=True,inferSchema=True)
df_shippers = spark.read.csv(f"{path_raw}shippers.csv",header=True,inferSchema=True)
df_suppliers = spark.read.csv(f"{path_raw}suppliers.csv",header=True,inferSchema=True)

df_all = [
    ("categories", df_categories),
    ("customers", df_customers),
    ("employees", df_employees),
    ("order_details", df_order_details),
    ("orders", df_orders),
    ("products", df_products),
    ("shippers", df_shippers),
    ("suppliers", df_suppliers)
]



In [4]:
print("Q11 — Valeurs nulles")
# print("Valeurs nulles de categories")
# for c in df_categories.columns:
#     nb_null = df_categories.filter(col(c).isNull()).count()
#     print(f"{c} : {nb_null} valeurs nulles")

print("Valeurs nulles par df")
for nom, df in df_all:
    print(f"\n{nom} : ")
    for c in df.columns:
        nb_null = df.filter(col(c).isNull()).count()
        print(f"{c} : {nb_null} valeurs nulles")

Q11 — Valeurs nulles
Valeurs nulles par df

categories : 
category_id : 0 valeurs nulles
category_name : 0 valeurs nulles
description : 0 valeurs nulles
picture : 8 valeurs nulles

customers : 
customer_id : 0 valeurs nulles
company_name : 0 valeurs nulles
contact_name : 0 valeurs nulles
contact_title : 0 valeurs nulles
address : 0 valeurs nulles
city : 0 valeurs nulles
region : 60 valeurs nulles
postal_code : 1 valeurs nulles
country : 0 valeurs nulles
phone : 0 valeurs nulles
fax : 22 valeurs nulles

employees : 
employee_id : 0 valeurs nulles
last_name : 0 valeurs nulles
first_name : 0 valeurs nulles
title : 0 valeurs nulles
title_of_courtesy : 0 valeurs nulles
birth_date : 0 valeurs nulles
hire_date : 0 valeurs nulles
address : 0 valeurs nulles
city : 0 valeurs nulles
region : 4 valeurs nulles
postal_code : 0 valeurs nulles
country : 0 valeurs nulles
home_phone : 0 valeurs nulles
extension : 0 valeurs nulles
photo : 9 valeurs nulles
notes : 0 valeurs nulles
reports_to : 1 valeurs n

In [5]:
# Q12 — Supprimer les nulls
# Dans df_orders, supprimer les lignes où shipped_date est null (commandes non livrées). Dans df_products,
#remplacer les valeurs nulles de unit_price par la médiane.

df_orders = df_orders.na.drop(subset=['shipped_date'])
median_price = df_products.agg({'unit_price': 'median'}).collect()[0][0]

df_products = df_products.fillna(
    value=median_price,
    subset=['unit_price']
)


In [6]:
# Q13 — Cast des types
#Dans df_orders, caster order_date, required_date et shipped_date en type DateType. Dans df_order_details,
#caster unit_price en DoubleType et quantity en IntegerType.
from pyspark.sql.types import DateType, DoubleType, IntegerType

print(df_orders.dtypes)

df_orders = df_orders.withColumn(
    'order_date',
    df_orders['order_date'].cast('date')
)

df_orders = df_orders.withColumn(
    'required_date',
    df_orders['required_date'].cast(DateType())
)

df_orders = df_orders.withColumn(
    'shipped_date',
    df_orders['shipped_date'].cast('date')
)

print(df_orders.dtypes)

df_order_details = df_order_details.withColumn(
    'unit_price',
    df_order_details['unit_price'].cast(DoubleType())
)

df_order_details = df_order_details.withColumn(
    'quantity',
    df_order_details['quantity'].cast(IntegerType())
)
print(df_order_details.dtypes)

[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('product_id', 'int'), ('unit_price', 'double'), ('quantity', 'int'), ('discount', 'double')]


In [7]:
# Q14 — Nettoyage des chaînes
#Dans df_customers, appliquer TRIM sur toutes les colonnes texte. Mettre contact_name en title case avec
#initcap(). Mettre country en majuscules avec upper().
from pyspark.sql import functions as F

df_customers = df_customers.select(
    *[F.trim(F.col(c)).alias(c) for c in df_customers.columns]
)

df_customers = df_customers.withColumn(
    "contact_name", F.initcap("contact_name")
).withColumn(
    "country", F.upper("country")
)
df_customers.show()




+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|region|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|      Berlin|  NULL|      12209|    GERMANY|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...| México D.F.|  NULL|      05021|     MEXICO|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312| México D.F.|  NULL|      05023|     MEXICO|  (5) 555-3

In [8]:
# Q15 — Renommer les colonnes
# Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite. Dans df_orders, renommer
# ship_via en shipper_id.

df_order_details = df_order_details.withColumnRenamed("unit_price","prix_unitaire")
df_order_details = df_order_details.withColumnRenamed("quantity","quantite")

df_orders = df_orders.withColumnRenamed("ship_via","shipper_id")


In [9]:
# Q16 — Colonnes calculées
# Dans df_order_details, ajouter une colonne sous_total = prix_unitaire * quantite * (1 - discount). Arrondir à 2
# décimales avec round().
from pyspark.sql import functions as F

df_order_details = df_order_details.withColumn(
    "sous_total",
        F.round(
            F.col("prix_unitaire") * F.col("quantite") * (1- F.col("discount")),
            2
        )
    )
df_order_details.show(1)

+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|
+--------+----------+-------------+--------+--------+----------+
only showing top 1 row


In [10]:
# Q17 — Colonnes conditionnelles
# Dans df_products, ajouter une colonne en_stock (True si units_in_stock > 0). Dans df_orders, ajouter une
# colonne is_shipped (True si shipped_date n'est pas null).

df_products =  df_products.withColumn(
    "en_stock",
    F.col("units_in_stock") > 0 
)

df_orders  = df_orders.withColumn(
    "is_shipped",
    F.col("shipped_date").isNotNull()
)
# df_products.show(1)
# df_orders.show(1)

In [11]:
# Q18 — Doublons
# Vérifier s'il y a des doublons dans df_customers sur customer_id. Utiliser distinct() et count() pour comparer.
# Supprimer les doublons si nécessaire.
print(df_customers.count())
print(df_customers.select("customer_id").distinct().count())



91
91


In [12]:
# Q19 — Filtrage
# Filtrer df_orders pour ne garder que les commandes de 1997. Filtrer df_products pour ne garder que les produits
# en stock (units_in_stock > 0) et non discontinués.
df_orders_1997 = df_orders.filter(
    F.year(F.col("order_date")) == 1997
)

In [16]:
# Q20 — Sélection de colonnes
# Dans df_employees, sélectionner uniquement : employee_id, first_name, last_name, title, hire_date, city,
# country. Créer une colonne full_name = first_name + ' ' + last_name.
from pyspark.sql.functions import concat, col, lit

df_employees_enrich = df_employees.select(
    col("employee_id"),
    col("first_name"),
    col("last_name"),
    col("title"),
    col("hire_date"),
    col("city"),
    col("country"),
    concat(col("first_name"), lit(" "), col("last_name")).alias("full_name")
)

df_employees_enrich.show(1)

+-----------+----------+---------+--------------------+----------+-------+-------+-------------+
|employee_id|first_name|last_name|               title| hire_date|   city|country|    full_name|
+-----------+----------+---------+--------------------+----------+-------+-------+-------------+
|          1|     Nancy|  Davolio|Sales Representative|1992-05-01|Seattle|    USA|Nancy Davolio|
+-----------+----------+---------+--------------------+----------+-------+-------+-------------+
only showing top 1 row


In [ ]:
spark.version

In [19]:
# Écrire les DataFrames nettoyés en Parquet : PATH = "/home/jovyan/data/tmp"
df_employees_enrich.write.format("parquet").save("/home/jovyan/data/tmp/employees_enrich.parquet")

In [20]:
df_customers.write.parquet("/home/jovyan/data/tmp/customers.parquet")

In [21]:
df_orders.write.parquet("/home/jovyan/data/tmp/orders.parquet")

In [22]:
df_products.write.parquet("/home/jovyan/data/tmp/products.parquet")

In [23]:
df_order_details.write.parquet("/home/jovyan/data/tmp/order_details.parquet")

In [24]:
df_categories.write.parquet("/home/jovyan/data/tmp/categories.parquet")

In [25]:
df_shippers.write.parquet("/home/jovyan/data/tmp/shippers.parquet")

In [26]:
df_suppliers.write.parquet("/home/jovyan/data/tmp/suppliers.parquet")